# Warp PR #1810 — CUDA verification of the svd2 fix

[NVIDIA/warp#1810](https://github.com/NVIDIA/warp/pull/1810) fixes GH-1734:
`wp.svd2` returns `U = V = I` for scaled orthogonal inputs. This notebook
verifies the fix **on CUDA** (the PR was developed CPU-only):
1. run the reproducer on the released `warp-lang` → bug visible on GPU,
2. build the PR branch from source (CUDA build, ~5–8 min) → reproducer clean,
3. run the full `test_mat.py` module on the GPU.

**Runtime → Change runtime type → T4 GPU**, then Run all (~10–15 min).

In [ ]:
!nvidia-smi -L && nvcc --version | tail -1
%pip install -q warp-lang numpy

In [ ]:
repro = r'''
import numpy as np, warp as wp

@wp.kernel
def decompose(a: wp.array(dtype=wp.mat22), rec: wp.array(dtype=wp.mat22)):
    tid = wp.tid()
    U, s, V = wp.svd2(a[tid])
    rec[tid] = U * wp.diag(s) * wp.transpose(V)

cases = {
    'rot(90deg)':  [[0.0, -1.0], [1.0, 0.0]],
    '-I':          [[-1.0, 0.0], [0.0, -1.0]],
    'diag(1,-1)':  [[1.0, 0.0], [0.0, -1.0]],
    '2*rot(0.5)':  [[2.0 * np.cos(0.5), -2.0 * np.sin(0.5)], [2.0 * np.sin(0.5), 2.0 * np.cos(0.5)]],
    'diag(2,2)  (GH-679)': [[2.0, 0.0], [0.0, 2.0]],
}
wp.init()
dev = 'cuda:0'
mats = [wp.mat22(*np.asarray(m, dtype=np.float32).flatten()) for m in cases.values()]
a = wp.array(mats, dtype=wp.mat22, device=dev)
rec = wp.zeros(len(mats), dtype=wp.mat22, device=dev)
wp.launch(decompose, dim=len(mats), inputs=[a], outputs=[rec], device=dev)
errs = np.abs(rec.numpy() - a.numpy()).reshape(len(mats), -1).max(axis=1)
print(f'warp {wp.config.version} on {dev}')
for name, e in zip(cases, errs):
    print(f'  {name:<22} max |U.diag(s).V^T - A| = {e:.3e}')
print('MAX_ERROR', errs.max())
'''
with open("repro_1734.py", "w") as f:
    f.write(repro)
print("reproducer written")

## 1. Released warp-lang on CUDA — expect the bug (errors up to 2.0)

In [ ]:
import subprocess, sys
before = subprocess.run([sys.executable, "repro_1734.py"], capture_output=True, text=True)
print(before.stdout, before.stderr[-500:])
err_before = float(before.stdout.split("MAX_ERROR")[1].split()[0])

## 2. Build the PR branch (CUDA build)

In [ ]:
!git clone -q --depth 1 --branch azrabano23/svd2-scaled-orthogonal https://github.com/azrabano23/warp.git warp-pr
!cd warp-pr && python build_lib.py 2>&1 | tail -5

In [ ]:
import os, subprocess, sys
env = dict(os.environ, PYTHONPATH=os.path.abspath("warp-pr"))
after = subprocess.run([sys.executable, "repro_1734.py"], capture_output=True, text=True, env=env)
print(after.stdout, after.stderr[-500:])
err_after = float(after.stdout.split("MAX_ERROR")[1].split()[0])

## 3. Full matrix test module on GPU (includes the new edge cases + gradient checks)

In [ ]:
tests = subprocess.run([sys.executable, "warp-pr/warp/tests/matrix/test_mat.py"],
                       capture_output=True, text=True, env=env)
print(tests.stdout[-2500:])
print(tests.stderr[-2500:])
tests_ok = tests.returncode == 0

In [ ]:
print("=" * 64)
verdict = err_before > 0.1 and err_after < 1e-5 and tests_ok
print("CUDA VERIFICATION OF PR #1810:", "PASSED" if verdict else "CHECK OUTPUT")
print(f"  reproducer max error  release: {err_before:.3e}   PR branch: {err_after:.3e}")
print(f"  test_mat.py on GPU: {'OK' if tests_ok else 'FAILED'}")
print("=" * 64)
print("Paste this block into the PR as a comment.")